In [1]:
!pip install --upgrade numpy pandas scikit-learn imbalanced-learn xgboost --user


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

# Load the preprocessed data from the preprocessing notebook
df = pd.read_csv('preprocessed_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (47886, 6)

Columns: ['statement', 'status', 'text_length', 'word_count', 'original_text', 'cleaned_text']

First few rows:


,statement,status,text_length,word_count,original_text,cleaned_text
0,"trouble sleeping, confused mind, restless hear...",Anxiety,64,10,"trouble sleeping, confused mind, restless hear...",trouble sleeping confused mind restless heart ...
1,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,78,14,"All wrong, back off dear, forward doubt. Stay ...",all wrong back dear forward doubt stay restles...
2,I've shifted my focus to something else but I'...,Anxiety,61,11,I've shifted my focus to something else but I'...,shifted focus something else still worried
3,"I'm restless and restless, it's been a month n...",Anxiety,72,14,"I'm restless and restless, it's been a month n...",restless restless month now boy what mean
4,"every break, you must be nervous, like somethi...",Anxiety,76,14,"every break, you must be nervous, like somethi...",every break must nervous like something wrong ...


In [3]:
# Check class distribution
print("Class Distribution:")
print("=" * 50)
class_dist = df['status'].value_counts()
print(class_dist)
print(f"\nPercentages:")
print((df['status'].value_counts(normalize=True) * 100).round(2))

# Check for any nulls in cleaned_text
print(f"\nNull values in cleaned_text: {df['cleaned_text'].isnull().sum()}")
print(f"Empty strings in cleaned_text: {(df['cleaned_text'] == '').sum()}")

Class Distribution:
status
Depression              15028
Normal                  13014
Suicidal                10602
Anxiety                  3554
Bipolar                  2500
Stress                   2293
Personality disorder      895
Name: count, dtype: int64

Percentages:
status
Depression              31.38
Normal                  27.18
Suicidal                22.14
Anxiety                  7.42
Bipolar                  5.22
Stress                   4.79
Personality disorder     1.87
Name: proportion, dtype: float64

Null values in cleaned_text: 0
Empty strings in cleaned_text: 0


In [4]:
!pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Run this cell ALONE before anything else
import sys
sys.path.insert(0, r'C:\users\3com\appdata\roaming\python\python310\site-packages')

# Fix numpy conflict
import numpy as np
print(f"Numpy version: {np.__version__}")
print(f"Numpy location: {np.__file__}")

Numpy version: 2.2.6
Numpy location: C:\Users\3com\AppData\Roaming\Python\Python310\site-packages\numpy\__init__.py


In [6]:
import sys
sys.path.append(r'C:\users\3com\appdata\roaming\python\python310\site-packages')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp

# Feature Extraction
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Class Imbalance
from imblearn.over_sampling import SMOTE

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [7]:
# Define X (input) and y (target)
X_text = df['cleaned_text'].astype(str)
y      = df['status']

# Encode labels to numbers (SVM, XGBoost need numeric labels)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes and their encoded values:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")

print(f"\nTotal samples: {len(X_text)}")
print(f"X shape: {X_text.shape}")
print(f"y shape: {y_encoded.shape}")

Classes and their encoded values:
  0 → Anxiety
  1 → Bipolar
  2 → Depression
  3 → Normal
  4 → Personality disorder
  5 → Stress
  6 → Suicidal

Total samples: 47886
X shape: (47886,)
y shape: (47886,)


In [8]:
# Split data BEFORE feature extraction to avoid data leakage
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded   # ensures same class distribution in train and test
)

print(f"Training samples: {len(X_train_text)}")
print(f"Testing samples:  {len(X_test_text)}")

print(f"\nClass distribution in training set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {le.classes_[cls]}: {cnt} ({cnt/len(y_train)*100:.1f}%)")

print(f"\nClass distribution in test set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {le.classes_[cls]}: {cnt} ({cnt/len(y_test)*100:.1f}%)")

Training samples: 38308
Testing samples:  9578

Class distribution in training set:
  Anxiety: 2843 (7.4%)
  Bipolar: 2000 (5.2%)
  Depression: 12022 (31.4%)
  Normal: 10411 (27.2%)
  Personality disorder: 716 (1.9%)
  Stress: 1834 (4.8%)
  Suicidal: 8482 (22.1%)

Class distribution in test set:
  Anxiety: 711 (7.4%)
  Bipolar: 500 (5.2%)
  Depression: 3006 (31.4%)
  Normal: 2603 (27.2%)
  Personality disorder: 179 (1.9%)
  Stress: 459 (4.8%)
  Suicidal: 2120 (22.1%)


In [9]:
# TF-IDF Vectorizer (unigrams + bigrams )
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),      # unigrams and bigrams
    max_features=50000,      # top 50k features to keep it manageable
    min_df=2,                # word must appear in at least 2 documents
    max_df=0.95,             # ignore words appearing in 95%+ of documents
    sublinear_tf=True        # apply log normalization to term frequency
)

# Fit on training data ONLY, then transform both
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print(f"TF-IDF Training matrix shape: {X_train_tfidf.shape}")
print(f"TF-IDF Testing matrix shape:  {X_test_tfidf.shape}")
print(f"\nExample top features:")
feature_names = tfidf.get_feature_names_out()
print(feature_names[:20])

TF-IDF Training matrix shape: (38308, 50000)
TF-IDF Testing matrix shape:  (9578, 50000)

Example top features:
['aampe' 'ab' 'aback' 'abandon' 'abandoned' 'abandoning' 'abandonment'
 'abandonment issue' 'abd' 'abdomen' 'abdominal' 'abdominal pain'
 'abilify' 'ability' 'ability feel' 'ability function' 'abit' 'able'
 'able accept' 'able achieve']


In [10]:
print(f"'original_text' column exists: {'original_text' in df.columns}")
print(f"Null values in original_text: {df['original_text'].isnull().sum()}")

'original_text' column exists: True
Null values in original_text: 0


In [11]:
# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(texts):
    """Extract 4 VADER scores for each text: pos, neg, neu, compound"""
    scores = []
    for text in texts:
        vs = analyzer.polarity_scores(text)
        scores.append([vs['pos'], vs['neg'], vs['neu'], vs['compound']])
    return np.array(scores)

# Use original_text for VADER (needs punctuation, capitals, emojis)
X_train_text_original = df.loc[X_train_text.index, 'original_text'].astype(str)
X_test_text_original  = df.loc[X_test_text.index,  'original_text'].astype(str)

print("Extracting VADER scores for training set...")
X_train_vader = get_vader_scores(X_train_text_original)

print("Extracting VADER scores for test set...")
X_test_vader  = get_vader_scores(X_test_text_original)

print(f"\nVADER Training matrix shape: {X_train_vader.shape}")
print(f"VADER Testing matrix shape:  {X_test_vader.shape}")
print(f"\nVADER score columns: [positive, negative, neutral, compound]")
print(f"Example first 3 rows:")
print(X_train_vader[:3])

Extracting VADER scores for training set...
Extracting VADER scores for test set...

VADER Training matrix shape: (38308, 4)
VADER Testing matrix shape:  (9578, 4)

VADER score columns: [positive, negative, neutral, compound]
Example first 3 rows:
[[ 0.158   0.228   0.614  -0.9652]
 [ 0.      0.155   0.845  -0.296 ]
 [ 0.06    0.118   0.823  -0.9232]]


In [12]:
# Convert VADER scores to sparse matrix to combine with TF-IDF
X_train_vader_sparse = sp.csr_matrix(X_train_vader)
X_test_vader_sparse  = sp.csr_matrix(X_test_vader)

# Horizontally stack TF-IDF + VADER features
X_train_combined = sp.hstack([X_train_tfidf, X_train_vader_sparse])
X_test_combined  = sp.hstack([X_test_tfidf,  X_test_vader_sparse])

print(f"TF-IDF features:           50,000")
print(f"VADER features:                 4")
print(f"─────────────────────────────────")
print(f"Combined Training shape: {X_train_combined.shape}")
print(f"Combined Testing shape:  {X_test_combined.shape}")

print(f"\n✓ Feature matrix ready for model training!")

TF-IDF features:           50,000
VADER features:                 4
─────────────────────────────────
Combined Training shape: (38308, 50004)
Combined Testing shape:  (9578, 50004)

✓ Feature matrix ready for model training!


In [13]:
# Apply SMOTE on training data ONLY (never on test data)
print("Applying SMOTE to handle class imbalance...")
print(f"Before SMOTE - Training samples: {X_train_combined.shape[0]}")
print(f"Before SMOTE - Class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {le.classes_[cls]}: {cnt}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)

print(f"\nAfter SMOTE - Training samples: {X_train_resampled.shape[0]}")
print(f"After SMOTE - Class distribution:")
unique, counts = np.unique(y_train_resampled, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {le.classes_[cls]}: {cnt}")

Applying SMOTE to handle class imbalance...
Before SMOTE - Training samples: 38308
Before SMOTE - Class distribution:
  Anxiety: 2843
  Bipolar: 2000
  Depression: 12022
  Normal: 10411
  Personality disorder: 716
  Stress: 1834
  Suicidal: 8482

After SMOTE - Training samples: 84154
After SMOTE - Class distribution:
  Anxiety: 12022
  Bipolar: 12022
  Depression: 12022
  Normal: 12022
  Personality disorder: 12022
  Stress: 12022
  Suicidal: 12022


In [14]:
import joblib
import os

# Create a folder to save all outputs
os.makedirs('feature_extraction_outputs', exist_ok=True)

# Save the feature matrices
joblib.dump(X_train_resampled, 'feature_extraction_outputs/X_train_resampled.pkl')
joblib.dump(y_train_resampled, 'feature_extraction_outputs/y_train_resampled.pkl')
joblib.dump(X_test_combined,   'feature_extraction_outputs/X_test_combined.pkl')
joblib.dump(y_test,            'feature_extraction_outputs/y_test.pkl')

# Save the TF-IDF vectorizer (teammates need it to transform new text)
joblib.dump(tfidf, 'feature_extraction_outputs/tfidf_vectorizer.pkl')

# Save the Label Encoder (teammates need it to decode predictions)
joblib.dump(le,    'feature_extraction_outputs/label_encoder.pkl')

# Save VADER analyzer
joblib.dump(analyzer, 'feature_extraction_outputs/vader_analyzer.pkl')

print("✓ All outputs saved to 'feature_extraction_outputs/' folder")
print("\nFiles saved:")
for f in os.listdir('feature_extraction_outputs'):
    size = os.path.getsize(f'feature_extraction_outputs/{f}') / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

print("\nTell your teammates to load like this:")
print("""
  import joblib
  X_train = joblib.load('feature_extraction_outputs/X_train_resampled.pkl')
  y_train = joblib.load('feature_extraction_outputs/y_train_resampled.pkl')
  X_test  = joblib.load('feature_extraction_outputs/X_test_combined.pkl')
  y_test  = joblib.load('feature_extraction_outputs/y_test.pkl')
  le      = joblib.load('feature_extraction_outputs/label_encoder.pkl')
""")

✓ All outputs saved to 'feature_extraction_outputs/' folder

Files saved:
  label_encoder.pkl — 0.0 MB
  tfidf_vectorizer.pkl — 1.9 MB
  vader_analyzer.pkl — 0.8 MB
  X_test_combined.pkl — 7.8 MB
  X_train_resampled.pkl — 129.6 MB
  y_test.pkl — 0.1 MB
  y_train_resampled.pkl — 0.6 MB

Tell your teammates to load like this:

  import joblib
  X_train = joblib.load('feature_extraction_outputs/X_train_resampled.pkl')
  y_train = joblib.load('feature_extraction_outputs/y_train_resampled.pkl')
  X_test  = joblib.load('feature_extraction_outputs/X_test_combined.pkl')
  y_test  = joblib.load('feature_extraction_outputs/y_test.pkl')
  le      = joblib.load('feature_extraction_outputs/label_encoder.pkl')

